# Research Paper RAG Pipeline
**Stack:** FastEmbed (`BAAI/bge-large-en-v1.5`) · Qdrant (local persistent) · Ollama (local LLM)

**Chunking strategy:** Section-aware splitting → sliding-window token chunking within each section

**Agentic features:** Multi-query decomposition · `ResearchRAG` tool wrapper

---
**Prerequisites:**
```bash
pip install -r requirements.txt
ollama pull llama3.2        # or mistral, phi4, deepseek-r1, etc.
ollama serve                # keep running in a separate terminal
```

In [ ]:
# Install if running in Colab / fresh env
# !pip install qdrant-client fastembed pymupdf ollama tqdm python-dotenv

## 1. Configuration

In [ ]:
import re
import uuid
from pathlib import Path
from typing import Optional

import fitz          # PyMuPDF
import numpy as np
from tqdm import tqdm

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)
import ollama

# ── Config ─────────────────────────────────────────────────────────────────
QDRANT_PATH     = "./qdrant_db"            # local persistent storage
COLLECTION_NAME = "research_papers"
EMBEDDING_MODEL = "BAAI/bge-large-en-v1.5" # 1024-dim, state-of-art retrieval
VECTOR_DIM      = 1024
OLLAMA_MODEL    = "llama3.2"               # change to mistral, phi4, deepseek-r1 …
CHUNK_SIZE      = 400   # target tokens per chunk  (bge-large max seq = 512)
CHUNK_OVERLAP   = 80    # token overlap between consecutive chunks
TOP_K           = 5     # chunks retrieved per query

## 2. PDF Processing & Section-Aware Chunking

Research papers have a predictable structure. We exploit that by:
1. **Detecting section headers** (numbered, ALL-CAPS, or known keywords).
2. **Chunking within each section** using a sliding-window token budget.

This keeps semantically related content together and lets us filter by section at query time.

In [ ]:
# Section header patterns — covers ACM, IEEE, arXiv, NeurIPS, ICML styles
_KNOWN_SECTIONS = (
    r"abstract|introduction|related work|background|literature review"
    r"|preliminaries|problem (statement|formulation)"
    r"|method(ology)?|approach|model|framework|architecture"
    r"|experiment(s|al setup)?|results?|evaluation|benchmark(s)?"
    r"|discussion|analysis|ablation"
    r"|conclusion(s)?|future work|limitations?"
    r"|acknowledgements?|references?|appendix|supplementary"
)

_SECTION_RE = re.compile(
    r"^\s*"
    r"(?:"
        r"(?:\d+\.)*\d+\.?\s+"       # numbered: "1.", "2.1"
        r"|[IVXLC]+\.\s+"             # roman: "III."
    r")?"
    r"(?:" + _KNOWN_SECTIONS + r")"
    r"\s*$",
    re.IGNORECASE,
)

# Also catch short ALL-CAPS lines (generic header fallback)
_ALLCAPS_RE = re.compile(r"^\s*[A-Z][A-Z\s\-]{3,55}$")


def _is_header(line: str) -> bool:
    stripped = line.strip()
    if not stripped or len(stripped) > 80:
        return False
    return bool(_SECTION_RE.match(stripped)) or bool(_ALLCAPS_RE.match(stripped))


def extract_sections(pdf_path: str) -> list[dict]:
    """Parse a research paper PDF into a list of section dicts."""
    doc  = fitz.open(pdf_path)
    title = Path(pdf_path).stem
    paper_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, str(Path(pdf_path).resolve())))

    # Collect lines with their page numbers
    lines: list[tuple[str, int]] = []
    for page_num, page in enumerate(doc, start=1):
        for line in page.get_text("text").split("\n"):
            lines.append((line, page_num))
    doc.close()

    sections: list[dict] = []
    cur_section = "preamble"
    cur_lines:  list[str] = []
    cur_pages:  list[int] = []

    for line, page_num in lines:
        if _is_header(line):
            if cur_lines:
                text = "\n".join(cur_lines).strip()
                if len(text) > 80:   # skip near-empty sections
                    sections.append({
                        "section":     cur_section,
                        "text":        text,
                        "pages":       sorted(set(cur_pages)),
                        "paper_title": title,
                        "paper_id":    paper_id,
                    })
            cur_section = line.strip().lower()
            cur_lines, cur_pages = [], []
        else:
            cur_lines.append(line)
            cur_pages.append(page_num)

    if cur_lines:
        text = "\n".join(cur_lines).strip()
        if len(text) > 80:
            sections.append({
                "section":     cur_section,
                "text":        text,
                "pages":       sorted(set(cur_pages)),
                "paper_title": title,
                "paper_id":    paper_id,
            })

    return sections

In [ ]:
def _word_to_token(n_words: int) -> int:
    """Rough conversion: avg English word ≈ 1.35 BPE tokens."""
    return int(n_words * 1.35)


def chunk_section(section: dict) -> list[dict]:
    """Sliding-window chunker that respects the bge-large 512-token limit."""
    words       = section["text"].split()
    word_limit  = int(CHUNK_SIZE  / 1.35)
    word_overlap = int(CHUNK_OVERLAP / 1.35)

    if len(words) <= word_limit:
        return [{
            **section,
            "chunk_index": 0,
            "chunk_id":    str(uuid.uuid4()),
            "n_tokens":    _word_to_token(len(words)),
        }]

    chunks, start, idx = [], 0, 0
    while start < len(words):
        end        = min(start + word_limit, len(words))
        chunk_text = " ".join(words[start:end])
        chunks.append({
            **section,
            "text":        chunk_text,
            "chunk_index": idx,
            "chunk_id":    str(uuid.uuid4()),
            "n_tokens":    _word_to_token(end - start),
        })
        start += word_limit - word_overlap
        idx   += 1
    return chunks


def process_paper(pdf_path: str) -> list[dict]:
    """Full pipeline: PDF → sections → chunks with metadata."""
    sections = extract_sections(pdf_path)
    chunks   = []
    for sec in sections:
        chunks.extend(chunk_section(sec))

    print(f"  [{Path(pdf_path).name}]  "
          f"{len(sections)} sections → {len(chunks)} chunks  "
          f"(avg {sum(c['n_tokens'] for c in chunks)//max(len(chunks),1)} tokens/chunk)")
    return chunks


# Quick sanity check — replace with an actual PDF path
# chunks = process_paper("path/to/attention_is_all_you_need.pdf")
# chunks[0]

## 3. Qdrant + FastEmbed Setup

In [ ]:
# FastEmbed downloads the ONNX model on first run (~600 MB for bge-large)
print("Loading embedding model …")
embed_model = TextEmbedding(model_name=EMBEDDING_MODEL)
print(f"Embedding model ready: {EMBEDDING_MODEL} ({VECTOR_DIM}-dim)")

# Local persistent Qdrant — data survives restarts
client = QdrantClient(path=QDRANT_PATH)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
    )
    print(f"Created collection '{COLLECTION_NAME}'")
else:
    info = client.get_collection(COLLECTION_NAME)
    print(f"Loaded collection '{COLLECTION_NAME}' — {info.points_count} vectors")

## 4. Indexing

In [ ]:
def index_paper(pdf_path: str, batch_size: int = 32):
    """Process a PDF and upsert all chunks into Qdrant."""
    chunks = process_paper(pdf_path)
    texts  = [c["text"] for c in chunks]

    print("  Embedding …")
    vectors = list(embed_model.embed(texts))   # returns numpy arrays

    points = [
        PointStruct(
            id=c["chunk_id"],
            vector=v.tolist(),
            payload={k: val for k, val in c.items() if k != "chunk_id"},
        )
        for c, v in zip(chunks, vectors)
    ]

    for i in tqdm(range(0, len(points), batch_size), desc="  Upserting", unit="batch"):
        client.upsert(collection_name=COLLECTION_NAME, points=points[i : i + batch_size])

    print(f"  Done — {len(points)} chunks indexed from '{Path(pdf_path).name}'\n")


def index_folder(folder: str, pattern: str = "*.pdf"):
    """Index every PDF in a folder."""
    pdfs = sorted(Path(folder).glob(pattern))
    print(f"Found {len(pdfs)} PDFs in '{folder}'")
    for pdf in pdfs:
        index_paper(str(pdf))


# ── Usage ──────────────────────────────────────────────────────────────────
# index_paper("papers/attention_is_all_you_need.pdf")
# index_folder("papers/")

## 5. Retrieval & RAG Generation

In [ ]:
def retrieve(
    query:           str,
    top_k:           int            = TOP_K,
    filter_paper_id: Optional[str]  = None,
    filter_section:  Optional[str]  = None,
) -> list[dict]:
    """Embed query and pull top-k chunks from Qdrant."""
    q_vec = list(embed_model.embed([query]))[0].tolist()

    must_conditions = []
    if filter_paper_id:
        must_conditions.append(FieldCondition(key="paper_id",    match=MatchValue(value=filter_paper_id)))
    if filter_section:
        must_conditions.append(FieldCondition(key="section",     match=MatchValue(value=filter_section)))

    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=q_vec,
        limit=top_k,
        query_filter=Filter(must=must_conditions) if must_conditions else None,
        with_payload=True,
    )

    return [
        {
            "score":       r.score,
            "text":        r.payload["text"],
            "section":     r.payload.get("section", "unknown"),
            "paper_title": r.payload.get("paper_title", "unknown"),
            "pages":       r.payload.get("pages", []),
            "chunk_index": r.payload.get("chunk_index", 0),
        }
        for r in results
    ]


def _format_context(chunks: list[dict]) -> str:
    parts = []
    for i, c in enumerate(chunks, 1):
        parts.append(
            f"[{i}] {c['paper_title']} | {c['section']} | p.{c['pages']} | score={c['score']:.3f}\n"
            f"{c['text']}"
        )
    return "\n\n---\n\n".join(parts)

In [ ]:
_SYSTEM_PROMPT = """You are a research assistant specialising in academic paper analysis.
Answer the question using ONLY the provided context chunks from research papers.
Cite sources using their chunk number [1], [2], etc.
If the context is insufficient to answer, say so explicitly."""


def rag_query(
    question:        str,
    top_k:           int           = TOP_K,
    filter_paper_id: Optional[str] = None,
    filter_section:  Optional[str] = None,
) -> str:
    """Single-query RAG: retrieve → generate."""
    chunks = retrieve(question, top_k=top_k,
                      filter_paper_id=filter_paper_id,
                      filter_section=filter_section)
    if not chunks:
        return "No relevant content found in the indexed papers."

    context  = _format_context(chunks)
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response["message"]["content"]

## 6. Agentic Layer — Multi-Query RAG + Tool Wrapper

Complex research questions often need multiple angles. The agentic layer:
1. **Decomposes** the question into focused sub-queries via the LLM.
2. **Retrieves** independently for each sub-query.
3. **Deduplicates & re-ranks** by cosine score.
4. **Generates** a single synthesised answer.

The `ResearchRAG` class wraps everything as a callable tool for any agent framework.

In [ ]:
def decompose_query(question: str) -> list[str]:
    """Ask the LLM to split a complex question into 2-3 targeted sub-queries."""
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You decompose complex research questions into 2-3 focused sub-queries "
                    "suitable for searching a vector database of paper chunks. "
                    "Reply with ONLY a Python list of strings — no explanation, no markdown."
                ),
            },
            {"role": "user", "content": question},
        ],
    )
    raw = resp["message"]["content"].strip()
    try:
        sub_qs = eval(raw)                       # LLM returns a list literal
        if isinstance(sub_qs, list) and sub_qs:
            return [str(q) for q in sub_qs]
    except Exception:
        pass
    return [question]                            # safe fallback


def multi_query_rag(question: str, top_k: int = TOP_K) -> str:
    """Agentic RAG: decompose → multi-retrieve → dedupe → generate."""
    sub_queries = decompose_query(question)
    print(f"Sub-queries: {sub_queries}")

    seen, all_chunks = set(), []
    for sq in sub_queries:
        for chunk in retrieve(sq, top_k=top_k):
            key = (chunk["paper_title"], chunk["section"], chunk["text"][:80])
            if key not in seen:
                seen.add(key)
                all_chunks.append(chunk)

    # Re-rank by score, keep top_k
    all_chunks = sorted(all_chunks, key=lambda x: x["score"], reverse=True)[:top_k]

    if not all_chunks:
        return "No relevant content found."

    context  = _format_context(all_chunks)
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response["message"]["content"]

In [ ]:
class ResearchRAG:
    """
    Drop-in tool for any agentic pipeline.

    Usage inside an agent:
        rag = ResearchRAG()
        answer = rag("What loss function did they use?")          # multi-query
        answer = rag("Summarise the abstract", multi_query=False)  # single-query
    """

    def __call__(
        self,
        question:    str,
        multi_query: bool          = True,
        top_k:       int           = TOP_K,
        paper_id:    Optional[str] = None,
        section:     Optional[str] = None,
    ) -> str:
        if multi_query:
            return multi_query_rag(question, top_k=top_k)
        return rag_query(question, top_k=top_k,
                         filter_paper_id=paper_id, filter_section=section)

    @staticmethod
    def add_paper(pdf_path: str):
        """Index a new paper into Qdrant."""
        index_paper(pdf_path)

    @staticmethod
    def add_folder(folder: str):
        """Index all PDFs in a folder."""
        index_folder(folder)

    @staticmethod
    def search(query: str, top_k: int = TOP_K) -> list[dict]:
        """Raw retrieval — returns chunk dicts with scores."""
        return retrieve(query, top_k=top_k)

    @staticmethod
    def collection_info() -> dict:
        info = client.get_collection(COLLECTION_NAME)
        return {"points": info.points_count, "collection": COLLECTION_NAME}


# Instantiate — hand `rag_tool` to your agent
rag_tool = ResearchRAG()
print("ResearchRAG tool ready.", rag_tool.collection_info())

## 7. Demo

Drop your PDFs in a `papers/` folder and run the cells below.

In [ ]:
# ── Index papers ───────────────────────────────────────────────────────────

# Single paper
# rag_tool.add_paper("papers/attention_is_all_you_need.pdf")

# Whole folder
# rag_tool.add_folder("papers/")

print(rag_tool.collection_info())

In [ ]:
# ── Simple query ───────────────────────────────────────────────────────────
# answer = rag_tool("What is the main contribution of the paper?", multi_query=False)
# print(answer)

In [ ]:
# ── Agentic multi-query ────────────────────────────────────────────────────
# answer = rag_tool(
#     "Compare the computational complexity and empirical results of the "
#     "proposed method against the baselines."
# )
# print(answer)

In [ ]:
# ── Section-filtered query (e.g. only look at methodology) ─────────────────
# answer = rag_tool(
#     "Describe the training procedure in detail.",
#     multi_query=False,
#     section="methodology",
# )
# print(answer)

In [ ]:
# ── Raw retrieval (inspect chunk scores) ───────────────────────────────────
# chunks = rag_tool.search("attention mechanism scaled dot product")
# for c in chunks:
#     print(f"[{c['score']:.3f}] {c['paper_title']} / {c['section']}")
#     print(c['text'][:200], "...\n")